# Online Retail - Data Cleaning

Raw dataset: UCI Online Retail II https://archive.ics.uci.edu/dataset/502/online+retail+ii

### Additional Variable Information

- InvoiceNo: Invoice number : 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. 
- StockCode: Product (item) code : 5-digit integral number uniquely assigned to each distinct product. 
- Description: Product (item) name 
- Quantity: The quantities of each product (item) per transaction. 	
- InvoiceDate: Invice date and time : The day and time when a transaction was generated. 
- UnitPrice: Unit price : Product price per unit in sterling (Â£). 
- CustomerID: Customer number : 5-digit integral number uniquely assigned to each customer. 
- Country: Country name : The name of the country where a customer resides.

In [1]:
import pandas as pd 

df = pd.read_excel('../data/raw_data/online_retail_II.xlsx', sheet_name=None)
df = pd.concat(df.values(), ignore_index=True)
df.shape

(1067371, 8)

### Data review for nulls, cancellations and other data issues

In [2]:
print(df.isnull().sum())

print(df['Invoice'].astype(str).str.startswith('C').sum(), "cancellation rows")


Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
19494 cancellation rows


### Data Cleaning Actions

- Customer ID : Dropped missing ~25% for segmentation analysis as known identifier needed for RFM. 
- Cancellations : Identified by invoice numbers that start with 'C' - Flagged to use in cancellation rate analysis
- Price <= 0 : Dropped from data as transactions are not valid
- Duplicate rows : Dropped exact duplicates


In [3]:
df['is_cancellation'] = df['Invoice'].astype(str).str.startswith('C')
df_clean = df.dropna(subset=['Customer ID']).copy()
df_clean = df_clean[df_clean['Price'] > 0]
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
df_clean = df_clean.drop_duplicates()
df_clean.shape

(797815, 10)

### Load data to Postgres

Writing cleaned table for SQL analysis

In [4]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
import pandas as pd

load_dotenv()

username = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
database = os.getenv('DB_NAME')

# Create the connection engine
engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Optional: only needed if you have a stale view from earlier work
with engine.connect() as conn:
    conn.execute(text("DROP VIEW IF EXISTS customer_segments"))
    conn.commit()

# Load DataFrame into PostgreSQL
table_name = "customer"
df_clean.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'customer' in database 'retail_portfolio'.
